In [ ]:
!pip -q install accelerate sentencepiece
!pip -q install pillow opencv-python torchvision
!pip -q install gradio
!pip install -U -q huggingface_hub==0.23.2 # Version compatible with transformers 4.44.2 and tokenizers
!pip install -U -q transformers==4.44.2 bitsandbytes flask flask-cors pyngrok flash_attn


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "5CD-AI/Vintern-1B-v2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=False
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    low_cpu_mem_usage=True
).eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Running on:", device)


In [ ]:
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB")),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])

def dynamic_preprocess(image, image_size=448, max_num=12):
    w, h = image.size
    ratio = w / h

    candidates = [(i, j) for i in range(1, max_num+1)
                          for j in range(1, max_num+1)
                          if i*j <= max_num]

    best = min(candidates, key=lambda x: abs(ratio - x[0]/x[1]))
    tw, th = best

    image = image.resize((tw*image_size, th*image_size))
    patches = []

    for i in range(tw*th):
        x = (i % tw) * image_size
        y = (i // tw) * image_size
        patches.append(image.crop((x, y, x+image_size, y+image_size)))

    patches.append(image.resize((image_size, image_size)))
    transform = build_transform(image_size)

    return torch.stack([transform(p) for p in patches])


In [ ]:
def vintern_agent(image_path, question, history=None):
    image = Image.open(image_path).convert("RGB")
    pixel_values = dynamic_preprocess(image).to(torch.bfloat16).to(device)

    generation_config = dict(
        max_new_tokens=1024,
        do_sample=False,
        num_beams=3,
        repetition_penalty=2.5
    )

    prompt = f"<image>\n{question}"

    response, history = model.chat(
        tokenizer,
        pixel_values,
        prompt,
        generation_config,
        history=history,
        return_history=True
    )
    return response


In [ ]:
RECEIPT_PROMPT = """
Bạn là AI Agent trích xuất thông tin từ hóa đơn / bill tiếng Việt.

Hãy đọc nội dung trong ảnh và trả về JSON hợp lệ theo schema sau.
Nếu không chắc chắn, hãy để null. Không bịa thông tin.

Schema JSON:
{
  "store_name": string | null,
  "branch": string | null,
  "receipt_id": string | null,
  "datetime": string | null,
  "items": [
    {"name": string, "quantity": number | null, "unit_price": number | null, "line_total": number | null}
  ],
  "subtotal": number | null,
  "tax": number | null,
  "total_amount": number | null,
  "currency": "VND" | "USD" | null,
  "confidence": "high" | "medium" | "low"
}

Chỉ trả về JSON, không giải thích.
"""


In [ ]:
import os, json, time, uuid

LOG_ROOT = "logs"
os.makedirs(LOG_ROOT, exist_ok=True)

def create_session():
    session_id = f"session_{int(time.time())}_{uuid.uuid4().hex[:6]}"
    session_dir = os.path.join(LOG_ROOT, session_id)
    os.makedirs(session_dir, exist_ok=True)
    return session_dir

def log_image(image, session_dir):
    path = os.path.join(session_dir, "input_image.png")
    image.save(path)
    return path

def log_request(question, session_dir):
    with open(os.path.join(session_dir, "request.json"), "w", encoding="utf-8") as f:
        json.dump({"question": question}, f, ensure_ascii=False, indent=2)

def log_prompt(prompt, session_dir):
    with open(os.path.join(session_dir, "agent_prompt.txt"), "w", encoding="utf-8") as f:
        f.write(prompt)

def log_raw_response(response, session_dir):
    with open(os.path.join(session_dir, "agent_response.txt"), "w", encoding="utf-8") as f:
        f.write(response)

def log_metadata(session_dir):
    with open(os.path.join(session_dir, "metadata.json"), "w", encoding="utf-8") as f:
        json.dump({
            "model": "Vintern-1B-v2",
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }, f, indent=2)


In [ ]:
def ui_handler(image, question):
    session_dir = create_session()

    image_path = log_image(image, session_dir)
    log_request(question, session_dir)

    full_prompt = RECEIPT_PROMPT + "\n\nCâu hỏi người dùng:\n" + question
    log_prompt(full_prompt, session_dir)

    response = vintern_agent(image_path, full_prompt)
    log_raw_response(response, session_dir)

    log_metadata(session_dir)

    return response


In [ ]:
import gradio as gr

with gr.Blocks(title="Vintern OCR Agent") as demo:
    gr.Markdown("## 🧾 AI Agent OCR tiếng Việt (Vintern-1B-v2)")
    gr.Markdown("Upload ảnh hóa đơn và đặt câu hỏi bằng tiếng Việt")

    with gr.Row():
        img = gr.Image(type="pil", label="Ảnh hóa đơn")
        txt = gr.Textbox(label="Câu hỏi", placeholder="Ví dụ: Tổng tiền là bao nhiêu?")

    btn = gr.Button("Hỏi AI Agent")
    out = gr.Textbox(label="Câu trả lời từ Agent", lines=12)

    btn.click(ui_handler, inputs=[img, txt], outputs=out)

demo.launch(share=True)


In [ ]:
!ls logs
